# UniProt Fetch

![UniProt Fetch](https://proto-bio.github.io/proto-assets/images/tool/uniprot/hero.png)

This notebook demonstrates `run_uniprot_fetch`, which provides a single-call interface to the UniProt REST API for retrieving protein entries. It supports two modes: direct lookup by accession, which is the fastest path when you already know the UniProt ID, and name-based search with organism filtering, which is useful when starting from a gene name in the literature. The search mode ranks candidates by gene name match quality, review status, and optionally by the presence of linked PDB structures, making it well-suited for workflows that need both a sequence and a structural template.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("uniprot")
display_overview("uniprot")
display_docs_section("uniprot", "Background")

# UniProt

UniProt (the Universal Protein Resource) is the reference database of protein sequences and their functional annotation, maintained by the UniProt consortium. The `uniprot-fetch` tool retrieves a UniProtKB entry over the UniProt REST API, either directly by accession or through a ranked gene- or protein-name and organism search, returning the sequence, gene names, review status, linked PDB structures, and the full JSON record. It runs on CPU and requires only network access.

UniProt ([The UniProt Consortium, 2025](https://doi.org/10.1093/nar/gkae1010)) is the central, freely accessible resource for protein sequence and functional annotation, maintained by [SIB](https://www.sib.swiss/), [EMBL-EBI](https://www.ebi.ac.uk/), and [PIR](https://proteininformationresource.org/). Its core database, UniProtKB, has two sections: [Swiss-Prot](https://en.wikipedia.org/wiki/UniProt#UniProtKB/Swiss-Prot), whose entries are manually reviewed and curated from the literature, and [TrEMBL](https://en.wikipedia.org/wiki/UniProt#UniProtKB/TrEMBL), whose entries are automatically annotated. As of release 2026_01 (January 2026), Swiss-Prot contains 574,627 reviewed entries, alongside hundreds of millions of unreviewed TrEMBL entries; current counts are published on the [UniProt statistics page](https://www.uniprot.org/uniprotkb/statistics).

Internally, the tool calls the UniProt REST API at `rest.uniprot.org`. Given an accession it fetches that UniProtKB entry directly; given a protein or gene name and an organism it runs a UniProt search and selects one entry deterministically, preferring an exact gene-name match, then optionally entries with linked PDB structures, then reviewed Swiss-Prot status. It extracts the sequence, length, review status, gene symbols, and PDB cross-references, and also returns the complete entry JSON; the `fields` option narrows the API response. Results reflect the live database at query time rather than a fixed release snapshot.

Records and their provenance come directly from UniProt's official REST API, maintained by the UniProt consortium.

## Available tools

In [2]:
display_available_tools("uniprot")

- **`run_uniprot_fetch()`** — Fetch protein entries from UniProt by accession or search by name and organism

### `run_uniprot_fetch`

`run_uniprot_fetch` fetches a protein entry from UniProt by accession or by searching with a gene name and organism. When given a UniProt accession directly, it retrieves the full entry in a single request. When given a name and organism, it issues two ranked search queries and selects the best candidate based on exact gene name match, reviewed (Swiss-Prot) status, and optionally PDB cross-reference availability. The returned output includes the primary accession, protein sequence, gene names, review status, linked PDB structure IDs, and the complete raw JSON entry for advanced access.

In [3]:
from proto_tools import (
    UniProtFetchConfig,
    UniProtFetchInput,
    run_uniprot_fetch,
)

In [4]:
# Display docs
display_api_reference("uniprot", "input", "run_uniprot_fetch")

# Fetch TP53 directly by its well-known UniProt accession
inputs = UniProtFetchInput(uniprot_id="P04637")

**Input** — `UniProtFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>uniprot_id</code> | <code>str &#124; None</code> | <code>None</code> | UniProt accession for direct entry lookup |
| <code>target_name</code> | <code>str &#124; None</code> | <code>None</code> | Gene or protein name for search |
| <code>organism</code> | <code>str &#124; None</code> | <code>None</code> | Organism for search disambiguation |
| <code>prefer_pdb_crossref</code> | <code>bool</code> | <code>False</code> | When searching, prefer entries that have linked PDB structures |
| <code>max_candidates</code> | <code>int</code> | <code>5</code> | Maximum number of search results to evaluate when ranking |

In [5]:
# Display config docs
display_api_reference("uniprot", "config", "run_uniprot_fetch")

# Use default config | see docs above for all fields
config = UniProtFetchConfig()

**Config** — `UniProtFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>fields</code> | <code>list[str] &#124; None</code> | <code>None</code> | Subset of UniProt fields to return; None = full entry. See www.uniprot.org/help/return_fields |

In [6]:
# Run the fetch
output = run_uniprot_fetch(inputs, config)

In [7]:
# Display output docs
display_api_reference("uniprot", "output", "run_uniprot_fetch")

# Inspect the retrieved TP53 entry
print(f"Accession: {output.accession}")
print(f"Entry type: {output.entry_type}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"Preview: {output.sequence[:60]}...")
print(f"PDB cross-refs: {output.pdb_crossrefs[:5]} ({len(output.pdb_crossrefs)} total)")
print(f"Source: {output.source_url}")

**Output** — `UniProtFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | Primary UniProt accession |
| <code>sequence</code> | <code>str &#124; None</code> | <code>None</code> | Protein sequence |
| <code>length</code> | <code>int &#124; None</code> | <code>None</code> | Sequence length |
| <code>entry_type</code> | <code>str &#124; None</code> | <code>None</code> | Review status (e.g. 'UniProtKB reviewed (Swiss-Prot)' for curated entries) |
| <code>gene_names</code> | <code>list[str]</code> | <code>[]</code> | Gene name symbols |
| <code>pdb_crossrefs</code> | <code>list[str]</code> | <code>[]</code> | PDB structure IDs linked to this protein entry |
| <code>source_url</code> | <code>str</code> | required | UniProt entry URL |
| <code>raw_entry</code> | <code>dict[str, Any]</code> | <code>{}</code> | Complete UniProt JSON record for advanced programmatic access |

Accession: P04637
Entry type: UniProtKB reviewed (Swiss-Prot)
Gene names: ['tp53']
Sequence length: 393
Preview: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...
PDB cross-refs: ['1A1U', '1AIE', '1C26', '1DT7', '1GZH'] (295 total)
Source: https://rest.uniprot.org/uniprotkb/P04637


#### Searching by Gene Name and Organism

When you do not have a UniProt accession, you can search by gene name and organism. The tool queries the UniProt search API, evaluates the top candidates, and ranks them by gene name match quality and review status. This is useful when starting from a gene name in the literature and needing to retrieve the canonical protein sequence.

In [8]:
# Search for lacI by gene name and organism
search_inputs = UniProtFetchInput(
    target_name="lacI",
    organism="Escherichia coli",
)

search_output = run_uniprot_fetch(search_inputs, config)

print(f"Accession: {search_output.accession}")
print(f"Gene names: {search_output.gene_names}")
print(f"Sequence length: {search_output.length}")
print(f"Preview: {search_output.sequence[:60]}...")

Accession: P03023
Gene names: ['laci']
Sequence length: 360
Preview: MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPNRVAQQLAGKQ...


#### Searching with PDB Cross-Reference Preference

Setting `prefer_pdb_crossref=True` prioritizes entries that have linked PDB structures during the ranking step. This is valuable when you need both a protein sequence and an experimental structure for downstream tasks like inverse folding or structure-guided design. Here we search for human insulin and increase the candidate pool to improve recall.

In [9]:
# Search for human insulin, preferring entries with PDB structures
pdb_inputs = UniProtFetchInput(
    target_name="insulin",
    organism="Homo sapiens",
    prefer_pdb_crossref=True,
    max_candidates=10,
)

pdb_output = run_uniprot_fetch(pdb_inputs, config)

print(f"Accession: {pdb_output.accession}")
print(f"Gene names: {pdb_output.gene_names}")
print(f"Sequence length: {pdb_output.length}")
if len(pdb_output.pdb_crossrefs) > 5:
    print(f"PDB cross-refs: {pdb_output.pdb_crossrefs[:5]}...")
else:
    print(f"PDB cross-refs: {pdb_output.pdb_crossrefs}")
print(f"Total PDB links: {len(pdb_output.pdb_crossrefs)}")

Accession: P01308
Gene names: ['ins']
Sequence length: 110
PDB cross-refs: ['1A7F', '1AI0', '1AIY', '1B9E', '1BEN']...
Total PDB links: 369


## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [10]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("uniprot_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'uniprot_results.json'}")

Exported to example_output/uniprot_results.json
